# M6 - AML - Assignment 2

## AutoML Using H20

## Business Context:

The study analyzes direct marketing campaigns conducted by a Portuguese banking institution to promote term deposits.

The objective is to predict whether a client will subscribe to a term deposit based on demographic, financial, and campaign-related information.

This predictive modeling helps optimize marketing efforts, reduce campaign costs, and improve customer targeting efficiency.

## Dataset Description

* age – Client’s age (numeric)
* job – Type of job (categorical)
* marital – Marital status (categorical)
* education – Education level (categorical)
* default – Has credit in default? (yes/no)
* balance – Average yearly account balance (numeric)
* housing – Has housing loan? (yes/no)
* loan – Has personal loan? (yes/no)
* contact – Contact communication type (telephone/cellular/unknown)
* duration – Call duration in seconds (numeric)
* campaign – Number of contacts during this campaign
* y – Whether the client subscribed to a term deposit (yes/no → binary classification)

## Questions:

* Initialize the H2O environment in Python.
* Import the training and test datasets into H2O.
* Convert the appropriate variables into categorical format.
* Train models using H2O AutoML.
* Retrieve the leaderboard and identify the best-performing model.
* Generate predictions on the test dataset using the best model.
* Explain the best-performing model.

In [77]:
import pandas as pd

train_df = pd.read_csv("Bank_Marketing_Train.csv")
train_df.head()

,age,job,marital,education,default,balance,housing,loan,contact,duration,campaign,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,261,1,no
1,44,technician,single,secondary,no,29,yes,no,unknown,151,1,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,76,1,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,92,1,no
4,33,unknown,single,unknown,no,1,no,no,unknown,198,1,no


In [78]:
train_df.dtypes

age          int64
job            str
marital        str
education      str
default        str
balance      int64
housing        str
loan           str
contact        str
duration     int64
campaign     int64
y              str
dtype: object

In [79]:
train_df.describe()

,age,balance,duration,campaign
count,45211.000000,45211.000000,45211.000000,45211.000000
mean,40.936210,1362.272058,258.163080,2.763841
std,10.618762,3044.765829,257.527812,3.098021
min,18.000000,-8019.000000,0.000000,1.000000
25%,33.000000,72.000000,103.000000,1.000000
50%,39.000000,448.000000,180.000000,2.000000
75%,48.000000,1428.000000,319.000000,3.000000
max,95.000000,102127.000000,4918.000000,63.000000


In [80]:
train_df.isnull().sum()

age          0
job          0
marital      0
education    0
default      0
balance      0
housing      0
loan         0
contact      0
duration     0
campaign     0
y            0
dtype: int64

In [81]:
test_df = pd.read_csv("Bank_Marketing_Test.csv")
test_df.head()

,age,job,marital,education,default,balance,housing,loan,contact,duration,campaign,y
0,30,unemployed,married,primary,no,1787,no,no,cellular,79,1,no
1,33,services,married,secondary,no,4789,yes,yes,cellular,220,1,no
2,35,management,single,tertiary,no,1350,yes,no,cellular,185,1,no
3,30,management,married,tertiary,no,1476,yes,yes,unknown,199,4,no
4,59,blue-collar,married,secondary,no,0,yes,no,unknown,226,1,no


In [82]:
test_df.dtypes

age          int64
job            str
marital        str
education      str
default        str
balance      int64
housing        str
loan           str
contact        str
duration     int64
campaign     int64
y              str
dtype: object

In [83]:
test_df.describe()

,age,balance,duration,campaign
count,4521.000000,4521.000000,4521.000000,4521.000000
mean,41.170095,1422.657819,263.961292,2.793630
std,10.576211,3009.638142,259.856633,3.109807
min,19.000000,-3313.000000,4.000000,1.000000
25%,33.000000,69.000000,104.000000,1.000000
50%,39.000000,444.000000,185.000000,2.000000
75%,49.000000,1480.000000,329.000000,3.000000
max,87.000000,71188.000000,3025.000000,50.000000


In [84]:
test_df.isnull().sum()

age          0
job          0
marital      0
education    0
default      0
balance      0
housing      0
loan         0
contact      0
duration     0
campaign     0
y            0
dtype: int64

In [85]:
import os

os.environ["JAVA_HOME"] = "/Users/niltonconstantino/.sdkman/candidates/java/11.0.31-amzn" # H2O used requires Python 3.11 and Java 11
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

import h2o

h2o.init()

Checking whether there is an H2O instance running at http://localhost:54321 . connected.


H2O_cluster_uptime:,1 hour 9 mins
H2O_cluster_timezone:,Europe/Dublin
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.30.0.4
H2O_cluster_version_age:,"5 years, 11 months and 4 days !!!"
H2O_cluster_name:,H2O_from_python_niltonconstantino_az1pjf
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,3.502 Gb
H2O_cluster_total_cores:,10
H2O_cluster_allowed_cores:,10
H2O_cluster_status:,"locked, healthy"


In [86]:
train_h2o = h2o.H2OFrame(train_df)
test_h2o = h2o.H2OFrame(test_df)

Parse progress: |█████████████████████████████████████████████████████████| 100%
Parse progress: |█████████████████████████████████████████████████████████| 100%


In [87]:
cols_to_factor = [
    "job",
    "marital",
    "education",
    "default",
    "housing",
    "loan",
    "contact",
    "y",
]

for frame in [train_h2o, test_h2o]:
    for col in cols_to_factor:
        if col in frame.columns:
            frame[col] = frame[col].asfactor()

In [88]:
train_h2o.types

{'age': 'int',
 'job': 'enum',
 'marital': 'enum',
 'education': 'enum',
 'default': 'enum',
 'balance': 'int',
 'housing': 'enum',
 'loan': 'enum',
 'contact': 'enum',
 'duration': 'int',
 'campaign': 'int',
 'y': 'enum'}

In [89]:
test_h2o.types

{'age': 'int',
 'job': 'enum',
 'marital': 'enum',
 'education': 'enum',
 'default': 'enum',
 'balance': 'int',
 'housing': 'enum',
 'loan': 'enum',
 'contact': 'enum',
 'duration': 'int',
 'campaign': 'int',
 'y': 'enum'}

In [90]:
target = "y"
features = [col for col in train_h2o.columns if col != target]

features

['age',
 'job',
 'marital',
 'education',
 'default',
 'balance',
 'housing',
 'loan',
 'contact',
 'duration',
 'campaign']

In [91]:
from h2o.automl import H2OAutoML

aml = H2OAutoML(max_models=20, seed=42, max_runtime_secs=400)
aml.train(x=features, y=target, training_frame=train_h2o)

AutoML progress: |
12:32:06.345: AutoML: XGBoost is not available; skipping it.

████████████████████████████████████████████████████████| 100%


## Leaderboard and Best Model

In [92]:
leaderboard = aml.leaderboard
leaderboard.head()

model_id,auc,logloss,aucpr,mean_per_class_error,rmse,mse
StackedEnsemble_AllModels_AutoML_20260506_123206,0.889045,0.25778,0.489577,0.226477,0.277958,0.0772605
GBM_1_AutoML_20260506_123206,0.88805,0.244754,0.482563,0.220429,0.275015,0.0756332
StackedEnsemble_BestOfFamily_AutoML_20260506_123206,0.887875,0.258647,0.488731,0.228877,0.278427,0.0775215
GBM_2_AutoML_20260506_123206,0.887464,0.245036,0.482601,0.22252,0.275137,0.0757005
GBM_grid__1_AutoML_20260506_123206_model_6,0.887405,0.245184,0.482556,0.225443,0.275153,0.0757094
GBM_3_AutoML_20260506_123206,0.886691,0.246216,0.478246,0.220394,0.276146,0.0762568
GBM_5_AutoML_20260506_123206,0.886589,0.244716,0.48146,0.228328,0.274729,0.0754759
GBM_grid__1_AutoML_20260506_123206_model_1,0.886545,0.246638,0.479604,0.236082,0.275281,0.0757794
GBM_4_AutoML_20260506_123206,0.885378,0.247525,0.482376,0.220867,0.27671,0.0765682
GBM_grid__1_AutoML_20260506_123206_model_4,0.883051,0.249773,0.469838,0.22626,0.277916,0.0772372


In [93]:
best_model = aml.leader
best_model.model_id

'StackedEnsemble_AllModels_AutoML_20260506_123206'

The best-performing model is the first model in the leaderboard, returned by `aml.leader`.

## Predictions on the Test Dataset

In [94]:
test_predictions = best_model.predict(test_h2o)
test_predictions.head()

stackedensemble prediction progress: |████████████████████████████████████| 100%


predict,no,yes
no,0.963531,0.036469
no,0.960945,0.0390549
no,0.952685,0.0473146
no,0.967828,0.0321716
no,0.967748,0.0322519
no,0.945266,0.0547337
no,0.93457,0.0654303
no,0.96288,0.0371204
no,0.968621,0.031379
no,0.964341,0.035659


In [95]:
test_results = test_h2o.cbind(test_predictions)
test_results.head()

age,job,marital,education,default,balance,housing,loan,contact,duration,campaign,y,predict,no,yes
30,unemployed,married,primary,no,1787,no,no,cellular,79,1,no,no,0.963531,0.036469
33,services,married,secondary,no,4789,yes,yes,cellular,220,1,no,no,0.960945,0.0390549
35,management,single,tertiary,no,1350,yes,no,cellular,185,1,no,no,0.952685,0.0473146
30,management,married,tertiary,no,1476,yes,yes,unknown,199,4,no,no,0.967828,0.0321716
59,blue-collar,married,secondary,no,0,yes,no,unknown,226,1,no,no,0.967748,0.0322519
35,management,single,tertiary,no,747,no,no,cellular,141,2,no,no,0.945266,0.0547337
36,self-employed,married,tertiary,no,307,yes,no,cellular,341,1,no,no,0.93457,0.0654303
39,technician,married,secondary,no,147,yes,no,cellular,151,2,no,no,0.96288,0.0371204
41,entrepreneur,married,tertiary,no,221,yes,no,unknown,57,2,no,no,0.968621,0.031379
43,services,married,primary,no,-88,yes,yes,cellular,313,1,no,no,0.964341,0.035659


The output includes the predicted class (`predict`) and the class probabilities (`p0` and `p1`) for each record in the test dataset.

## Explain the Best-Performing Model

In [96]:
best_model

Model Details
H2OStackedEnsembleEstimator :  Stacked Ensemble
Model Key:  StackedEnsemble_AllModels_AutoML_20260506_123206

No model summary for this model

ModelMetricsBinomialGLM: stackedensemble
** Reported on train data. **

MSE: 0.03935389015675662
RMSE: 0.19837814939341636
LogLoss: 0.14784413032306068
Null degrees of freedom: 9970
Residual degrees of freedom: 9963
Null deviance: 7057.260930230174
Residual deviance: 2948.307646902476
AIC: 2964.307646902476
AUC: 0.9810767270972645
AUCPR: 0.8835408442797298
Gini: 0.9621534541945289

Confusion Matrix (Act/Pred) for max f1 @ threshold = 0.3238355166429959: 


,,no,yes,Error,Rate
0,no,8621.0,218.0,0.0247,(218.0/8839.0)
1,yes,284.0,848.0,0.2509,(284.0/1132.0)
2,Total,8905.0,1066.0,0.0503,(502.0/9971.0)



Maximum Metrics: Maximum metrics at their respective thresholds


,metric,threshold,value,idx
0,max f1,0.323836,0.771611,194.0
1,max f2,0.121004,0.843095,285.0
2,max f0point5,0.482323,0.832927,140.0
3,max accuracy,0.382433,0.950155,174.0
4,max precision,0.943774,1.000000,0.0
5,max recall,0.060734,1.000000,338.0
6,max specificity,0.943774,1.000000,0.0
7,max absolute_mcc,0.323836,0.743772,194.0
8,max min_per_class_accuracy,0.159925,0.925105,265.0
9,max mean_per_class_accuracy,0.112008,0.935286,291.0



Gains/Lift Table: Avg response rate: 11.35 %, avg score: 11.85 %


...

,,no,yes,Error,Rate
0,no,35773.0,4149.0,0.1039,(4149.0/39922.0)
1,yes,1846.0,3443.0,0.349,(1846.0/5289.0)
2,Total,37619.0,7592.0,0.1326,(5995.0/45211.0)



Maximum Metrics: Maximum metrics at their respective thresholds


,metric,threshold,value,idx
0,max f1,0.175272,0.534586,267.0
1,max f2,0.076823,0.655584,332.0
2,max f0point5,0.360841,0.522050,183.0
3,max accuracy,0.551377,0.892924,113.0
4,max precision,0.928270,1.000000,0.0
5,max recall,0.031378,1.000000,398.0
6,max specificity,0.928270,1.000000,0.0
7,max absolute_mcc,0.128596,0.475945,295.0
8,max min_per_class_accuracy,0.084984,0.812441,325.0
9,max mean_per_class_accuracy,0.074743,0.815913,334.0



Gains/Lift Table: Avg response rate: 11.70 %, avg score: 11.70 %


,,group,cumulative_data_fraction,lower_threshold,lift,cumulative_lift,response_rate,score,cumulative_response_rate,cumulative_score,capture_rate,cumulative_capture_rate,gain,cumulative_gain
0,,1,0.010020,0.799826,5.340215,5.340215,0.624724,0.845005,0.624724,0.845005,0.053507,0.053507,434.021546,434.021546
1,,2,0.020017,0.737062,5.370942,5.355562,0.628319,0.767150,0.626519,0.806121,0.053696,0.107204,437.094186,435.556168
2,,3,0.030015,0.686440,5.219648,5.310290,0.610619,0.711876,0.621223,0.774729,0.052184,0.159387,421.964772,431.029042
3,,4,0.040012,0.631853,4.557736,5.122256,0.533186,0.659348,0.599226,0.745900,0.045566,0.204954,355.773588,412.225578
4,,5,0.050010,0.581431,4.917059,5.081235,0.575221,0.606927,0.594427,0.718117,0.049159,0.254112,391.705945,408.123467
5,,6,0.100020,0.354014,4.136065,4.608650,0.483857,0.462155,0.539142,0.590136,0.206844,0.460957,313.606453,360.864960
6,,7,0.150008,0.208568,2.806506,4.008112,0.328319,0.273451,0.468888,0.484605,0.140291,0.601248,180.650624,300.811229
7,,8,0.200018,0.130202,2.412074,3.609059,0.282176,0.164840,0.422205,0.404655,0.120628,0.721876,141.207419,260.905865
8,,9,0.300015,0.067820,1.350001,2.856095,0.157930,0.092192,0.334120,0.300508,0.134997,0.856873,35.000150,185.609511
9,,10,0.400013,0.047748,0.722270,2.322668,0.084495,0.056178,0.271717,0.239429,0.072225,0.929098,-27.773029,132.266826


In [97]:
test_perf = best_model.model_performance(test_data=test_h2o)
test_perf


ModelMetricsBinomialGLM: stackedensemble
** Reported on test data. **

MSE: 0.04321635921559237
RMSE: 0.20788544733961628
LogLoss: 0.15818394672515373
Null degrees of freedom: 4520
Residual degrees of freedom: 4513
Null deviance: 3231.134056940978
Residual deviance: 1430.2992462888403
AIC: 1446.2992462888403
AUC: 0.9758397312859884
AUCPR: 0.8558724731209615
Gini: 0.9516794625719769

Confusion Matrix (Act/Pred) for max f1 @ threshold = 0.3138334250013924: 


,,no,yes,Error,Rate
0,no,3889.0,111.0,0.0278,(111.0/4000.0)
1,yes,143.0,378.0,0.2745,(143.0/521.0)
2,Total,4032.0,489.0,0.0562,(254.0/4521.0)



Maximum Metrics: Maximum metrics at their respective thresholds


,metric,threshold,value,idx
0,max f1,0.313833,0.748515,191.0
1,max f2,0.155833,0.827905,269.0
2,max f0point5,0.508095,0.797060,128.0
3,max accuracy,0.330271,0.943818,185.0
4,max precision,0.928911,1.000000,0.0
5,max recall,0.071585,1.000000,324.0
6,max specificity,0.928911,1.000000,0.0
7,max absolute_mcc,0.313833,0.717383,191.0
8,max min_per_class_accuracy,0.161628,0.915250,266.0
9,max mean_per_class_accuracy,0.116450,0.926685,291.0



Gains/Lift Table: Avg response rate: 11.52 %, avg score: 12.04 %


,,group,cumulative_data_fraction,lower_threshold,lift,cumulative_lift,response_rate,score,cumulative_response_rate,cumulative_score,capture_rate,cumulative_capture_rate,gain,cumulative_gain
0,,1,0.010175,0.859912,8.677543,8.677543,1.000000,0.886311,1.000000,0.886311,0.088292,0.088292,767.754319,767.754319
1,,2,0.020128,0.827664,8.677543,8.677543,1.000000,0.842317,1.000000,0.864556,0.086372,0.174664,767.754319,767.754319
2,,3,0.030082,0.782293,8.677543,8.677543,1.000000,0.806965,1.000000,0.845500,0.086372,0.261036,767.754319,767.754319
3,,4,0.040035,0.721436,8.484709,8.629601,0.977778,0.754344,0.994475,0.822837,0.084453,0.345489,748.470889,762.960096
4,,5,0.050210,0.644326,7.922974,8.486408,0.913043,0.682769,0.977974,0.794453,0.080614,0.426104,692.297421,748.640787
5,,6,0.100199,0.344379,5.068300,6.781126,0.584071,0.476614,0.781457,0.635884,0.253359,0.679463,406.829956,578.112646
6,,7,0.150188,0.210308,3.186885,5.584811,0.367257,0.267082,0.643594,0.513131,0.159309,0.838772,218.688533,458.481056
7,,8,0.200177,0.132663,2.111791,4.717515,0.243363,0.169665,0.543646,0.427360,0.105566,0.944338,111.179148,371.751519
8,,9,0.300155,0.065263,0.556745,3.331614,0.064159,0.087993,0.383935,0.314321,0.055662,1.000000,-44.325497,233.161385
9,,10,0.400133,0.044897,0.000000,2.499171,0.000000,0.053097,0.288004,0.249051,0.000000,1.000000,-100.000000,149.917081


In [99]:
print("Base models used in the ensemble:")
print(best_model.base_models)

Base models used in the ensemble:
['GBM_1_AutoML_20260506_123206', 'GBM_2_AutoML_20260506_123206', 'GBM_grid__1_AutoML_20260506_123206_model_6', 'GBM_3_AutoML_20260506_123206', 'GBM_5_AutoML_20260506_123206', 'GBM_grid__1_AutoML_20260506_123206_model_1', 'GBM_4_AutoML_20260506_123206', 'GBM_grid__1_AutoML_20260506_123206_model_4', 'GBM_grid__1_AutoML_20260506_123206_model_2', 'XRT_1_AutoML_20260506_123206', 'GBM_grid__1_AutoML_20260506_123206_model_3', 'GBM_grid__1_AutoML_20260506_123206_model_5', 'DRF_1_AutoML_20260506_123206', 'GLM_1_AutoML_20260506_123206', 'DeepLearning_1_AutoML_20260506_123206', 'DeepLearning_grid__1_AutoML_20260506_123206_model_1', 'DeepLearning_grid__1_AutoML_20260506_123206_model_2', 'DeepLearning_grid__3_AutoML_20260506_123206_model_1', 'DeepLearning_grid__2_AutoML_20260506_123206_model_1', 'DeepLearning_grid__3_AutoML_20260506_123206_model_2']


Since the leader is a Stacked Ensemble, which means H2O combined several strong base models to improve prediction accuracy. Most base models are GBMs, suggesting that tree-based boosting performs particularly well on this bank marketing dataset.